In [ ]:
import os, sys, pathlib
# Work from the repo root: find the folder containing `src/`, put it on the import path,
# and chdir into it so imports AND relative paths (configs/, data/, outputs/) resolve the
# same as running a script from the project root.
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Drop any stale/namespace `src`/`scripts` cached by an earlier failed import.
for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

print('working dir:', ROOT)

In [ ]:
from src.predict import run_all
from src.config import resolve_config_path, Config
CONDITION = "iq_imbalance_100"
cfg = Config.from_yaml(resolve_config_path(CONDITION))

run_all(cfg)

In [ ]:
from pathlib import Path
import numpy as np
from src.data import MODULATION_CLASSES

RUNS = Path("runs")
files = sorted(RUNS.glob(f"{CONDITION}/*/predictions.npz"))
print(len(files), "cells")
for f in files:
    print(" ", f.parent.relative_to(RUNS))

In [ ]:
d = np.load(files[0])
pred, true, snr = d["pred"], d["true"], d["snr"]
print(pred.shape, true.shape, snr.shape)
print("класи в true:", np.unique(true).size, "з", len(MODULATION_CLASSES))
print("класи в pred:", np.unique(pred).size)
print("SNR:", np.unique(snr))

In [ ]:
acc = (pred == true).mean()
mask = snr >= 0
print(f"усі кадри:   {acc:.4f}")
print(f"SNR >= 0 dB: {(pred[mask] == true[mask]).mean():.4f}")

In [ ]:
for s in np.unique(snr):
    m = snr == s
    print(f"{s:>4} dB  n={m.sum():>6}  acc={(pred[m] == true[m]).mean():.3f}")

In [ ]:
for c in range(len(MODULATION_CLASSES)):
    m = mask & (true == c)
    if m.sum():
        print(f"{MODULATION_CLASSES[c]:>10}  n={m.sum():>5}  recall={(pred[m] == c).mean():.3f}")

In [ ]:
for f in files:
    z = np.load(f)
    m = z["snr"] >= 0
    print(f.parent.name, f"{(z['pred'][m] == z['true'][m]).mean():.4f}")

In [ ]:
from pathlib import Path
import numpy as np
from src.data import MODULATION_CLASSES

RUNS = Path("runs")
runs = {}
for f in sorted(RUNS.glob(f"{CONDITION}/*/predictions.npz")):
    z = np.load(f)
    m = z["snr"] >= 0
    runs[f.parent.name] = {"pred": z["pred"][m], "true": z["true"][m]}

names = list(runs)
C = len(MODULATION_CLASSES)

In [ ]:
recall = np.array([
    [(runs[n]["pred"][runs[n]["true"] == c] == c).mean() for n in names]
    for c in range(C)
])                                          # (класи, зерна)

print(f"{'class':>10} " + " ".join(f"{n:>7}" for n in names) + "   СКВ")
for c in range(C):
    print(f"{MODULATION_CLASSES[c]:>10} "
          + " ".join(f"{recall[c, j]:7.3f}" for j in range(len(names)))
          + f"  {recall[c].std(ddof=1):.3f}")

In [ ]:
from itertools import combinations

recall_by_seed = {}
for f in sorted(RUNS.glob(f"{CONDITION}/seed*/predictions.npz")):
    z = np.load(f)
    m = z["snr"] >= 0
    pred, true = z["pred"][m], z["true"][m]
    seed = int(f.parent.name[len("seed"):])
    recall_by_seed[seed] = np.array([(pred[true == c] == c).mean() for c in range(C)])

seeds = sorted(recall_by_seed)
print(CONDITION, "seeds:", seeds)

for j in seeds[1:4]:
    d = recall_by_seed[seeds[0]] - recall_by_seed[j]
    worst = int(np.abs(d).argmax())
    print(f"{seeds[0]} vs {j}:  mean|d| = {100 * np.abs(d).mean():5.2f} pp"
          f"   max|d| = {100 * np.abs(d).max():5.2f} pp ({MODULATION_CLASSES[worst]})")

pairs = list(combinations(seeds, 2))
m_ij = np.array([np.abs(recall_by_seed[i] - recall_by_seed[j]).mean() for i, j in pairs])
print(f"\n{len(pairs)} pairs:  median {100 * np.median(m_ij):.2f} pp"
      f"   min {100 * m_ij.min():.2f} pp   max {100 * m_ij.max():.2f} pp")

In [ ]:
import matplotlib.pyplot as plt

values = 100 * m_ij
median = float(np.median(values))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.hist(values, bins=12, color="#3d63dd", edgecolor="white", linewidth=1.2)
ax.axvline(median, color="#3a3f45", lw=2, ls="--")
ax.annotate(f"median {median:.2f} pp", xy=(median, ax.get_ylim()[1]),
            xytext=(6, -10), textcoords="offset points",
            fontsize=9, color="#3a3f45", va="top")
ax.set_xlabel("mean |recall$_i$ - recall$_j$| over 24 classes (pp)")
ax.set_ylabel(f"pairs (n = {len(pairs)})")
ax.set_title(f"{CONDITION}: seed-to-seed per-class recall distance, SNR >= 0 dB")
ax.grid(axis="y", alpha=0.25, lw=0.6)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
plt.show()